In [ ]:
# Build FAISS Index for Knowledge Base
This notebook creates a searchable index of knowledge base embeddings using FAISS (Facebook AI Similarity Search).

**Purpose:**
- Encode all knowledge base subjects into 128-dimensional embeddings
- Build a FAISS index for efficient nearest neighbor retrieval
- This index enables the RAC model to retrieve similar cases during classification

import os
import torch
import pandas as pd
import numpy as np
import faiss  # Facebook AI Similarity Search library for efficient vector retrieval
from torch_geometric.loader import DataLoader
from models import GraphAutoencoder
from utils import FCDataset

# ===== DEFINE FILE PATHS =====
output_dir = r'C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs'
master_csv = os.path.join(output_dir, "master_metadata.csv")
encoder_weights = os.path.join(output_dir, 'gae_encoder.pth')  # Trained encoder from Script 02
index_save_path = os.path.join(output_dir, 'knowledge_base.index')  # Where to save FAISS index

# ===== LOAD AND FILTER METADATA =====
# Load master metadata and filter for knowledge base datasets
df = pd.read_csv(master_csv)
df_kb = df[df['dataset_source'].isin(['PPMI', 'Neurocon'])].reset_index(drop=False)

# Preserve original master index for tracking which subjects are in the index
df_kb = df_kb.rename(columns={'index': 'original_master_index'})

# ===== CREATE DATALOADER =====
# Create dataset and dataloader for knowledge base
# shuffle=False: Keep subjects in order so indices match metadata rows
kb_dataset = FCDataset(df_kb)
kb_dataloader = DataLoader(kb_dataset, batch_size=32, shuffle=False)

# ===== LOAD TRAINED ENCODER =====
# Initialize a new GraphAutoencoder with the same architecture
model = GraphAutoencoder(num_nodes=100, input_dim=100, hidden_dim=64, embedding_dim=128)

# Load the trained encoder weights from Script 02
# This gives the encoder its learned "knowledge" about brain connectivity patterns
model.encoder.load_state_dict(torch.load(encoder_weights))

# Set to evaluation mode
# This "freezes" the weights and disables dropout/batch norm training behavior
model.encoder.eval()

# ===== ENCODE KNOWLEDGE BASE =====
print("encoding Knowledge Base into vector space...")
all_embeddings = []

# Disable gradient computation for efficiency (we're not training)
with torch.no_grad():
    for data, label in kb_dataloader:
        # ===== GENERATE EMBEDDINGS =====
        # Pass each brain graph through the encoder
        # Input: node features, edge connections, edge weights, batch assignment
        # Output: 128-dimensional embedding vector per subject
        # data.batch tells the encoder where one brain ends and the next begins
        v_embedding = model.encoder(data.x, data.edge_index, data.edge_weight, data.batch)
        
        # Move embeddings to CPU and convert to NumPy
        all_embeddings.append(v_embedding.cpu().numpy())

# ===== CONSOLIDATE EMBEDDINGS =====
# Concatenate all batches into a single array
# Shape: (num_subjects, 128)
all_embeddings = np.concatenate(all_embeddings, axis=0).astype('float32')  # FAISS requires float32

# ===== SAVE RAW EMBEDDINGS =====
# Save embeddings as NumPy array for later use in RAC model
# The RAC model needs access to actual embedding vectors (not just indices)
np.save(os.path.join(output_dir, 'kb_embeddings.npy'), all_embeddings)

# ===== CREATE FAISS INDEX =====
EMBEDDING_DIM = 128  # Dimensionality of embeddings

# Initialize FAISS index with L2 (Euclidean) distance
# IndexFlatL2: Stores vectors as-is without compression
# "Flat" means exact search (no approximation)
# "L2" means Euclidean distance metric
index = faiss.IndexFlatL2(EMBEDDING_DIM)

# Add all knowledge base embeddings to the index
# This creates a searchable database of brain embeddings
index.add(all_embeddings)

# Why IndexFlatL2?
# Since the knowledge base isn't too large, we use exact search for accuracy
# For larger databases, approximate methods (e.g., IndexIVFFlat) would be faster

# ===== SAVE INDEX AND METADATA =====
# Save FAISS index to disk for later retrieval
faiss.write_index(index, index_save_path)

# Save metadata with original indices preserved
# This allows us to map retrieved indices back to subject IDs
df_kb.to_csv(os.path.join(output_dir, "kb_metadata_indexed.csv"), index=False)

print(f"Index successfully saved to: {output_dir}")

# ===== VERIFICATION =====
print(f"--- Index Exploration ---")
print(f"Total vectors stored: {index.ntotal}")  # Should match number of KB subjects
print(f"Vector dimension: {index.d}")  # Should be 128

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import faiss
from torch_geometric.loader import DataLoader
from models import GraphAutoencoder
from utils import FCDataset


In [ ]:
output_dir = r'C:\Users\nikna\Documents\pp_datasets\processed_data_2'
master_csv = os.path.join(output_dir, "master_metadata.csv")
encoder_weights = os.path.join(output_dir, 'gae_encoder.pth')
index_save_path = os.path.join(output_dir, 'knowledge_base.index')

# 2. Load the metadata and filter for the knowledge base (like in script 02).
#    - df_kb = ...

df = pd.read_csv(master_csv)
df_kb = df[df['dataset_source'].isin(['PPMI', 'Neurocon'])].reset_index(drop=False)
df_kb = df_kb.rename(columns={'index': 'original_master_index'})

# 3. Create a DataLoader (no shuffling needed)
#    - kb_dataset = FCDataset(df_kb)
#    - kb_dataloader = DataLoader(kb_dataset, batch_size=32, shuffle=False)

kb_dataset = FCDataset(df_kb)
kb_dataloader = DataLoader(kb_dataset, batch_size=32, shuffle=False)


In [8]:
model = GraphAutoencoder(num_nodes=100, input_dim=100, hidden_dim=64, embedding_dim=128)
model.encoder.load_state_dict(torch.load(encoder_weights)) # this gives the model its knowledge (weights)
model.encoder.eval() # this "freezes" that knowledge while building the index

GAEEncoder(
  (conv1): GCNConv(100, 64)
  (conv2): GCNConv(64, 64)
  (lin_encode): Linear(in_features=64, out_features=128, bias=True)
)

In [9]:
print("encoding Knowledge Base into vector space...")
all_embeddings = []

with torch.no_grad():
    for data, label in kb_dataloader:
        v_embedding = model.encoder(data.x, data.edge_index, data.edge_weight, data.batch) # without passing data.batch, the encoder wouldn't know where one brain ends and the next begins.
        all_embeddings.append(v_embedding.cpu().numpy())

all_embeddings = np.concatenate(all_embeddings, axis=0).astype('float32') # FAISS needs float32

# Save raw embeddings for the RAC model (Script 04 needs this to retrieve vectors)
np.save(os.path.join(output_dir, 'kb_embeddings.npy'), all_embeddings)

# 6. Create and populate the FAISS index
#    - EMBEDDING_DIM = 128
#    - index = faiss.IndexFlatL2(EMBEDDING_DIM) # (L2 distance = dot product for normalized vectors)
#    - index.add(all_embeddings)

EMBEDDING_DIM = 128
index = faiss.IndexFlatL2(EMBEDDING_DIM) # (L2 distance = dot product for normalized vectors)
index.add(all_embeddings)
# "IndexFlatL2" tells FAISS to store vectors as-is, without compressing or clustering them.
# since the knowledge base isn't too large, this is a good method because it performs an exact search rather than an approximation.
# L2 => Euclidean distance

# 7. Save the index to disk
#    - (e.g., faiss.write_index(index, 'knowledge_base.index'))

faiss.write_index(index, index_save_path)
df_kb.to_csv(os.path.join(output_dir, "kb_metadata_indexed.csv"), index=False)

print(f"Index successfully saved to: {output_dir}")

# Exploration / Verification
print(f"--- Index Exploration ---")
print(f"Total vectors stored: {index.ntotal}")
print(f"Vector dimension: {index.d}")

encoding Knowledge Base into vector space...
Index successfully saved to: C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs
--- Index Exploration ---
Total vectors stored: 236
Vector dimension: 128
